##Setup

In [1]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-o268axoo
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-o268axoo
  Resolved https://github.com/huggingface/transformers.git to commit e40b0c0195569becf0cafa63c21df33defa710f7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install -U accelerate torch torchvision --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [3]:
GEMMA_ID = "google/gemma-4-E2B-it"
LLAMA_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"
QWEN_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

In [4]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

# Load model
processor = AutoProcessor.from_pretrained(GEMMA_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    GEMMA_ID,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [5]:
PROMPT = """
Act as a Financial Data Extraction Agent.
Analyze the receipt image and provide a JSON response.

### LOGIC RULES:
1. EXCLUSIONS: Ignore payment methods (Visa, Cash, Change, etc.), membership numbers, transaction IDs, barcodes, and "Total Number of Items Sold" lines.
2. LINE ITEMS: Extract ONLY purchased items. For each item, extract [item name, quantity, unit_price, total_price, box_2d]. The box_2d should be a bounding box [ymin, xmin, ymax, xmax] normalized to 0-1000. If quantity is not explicitly stated, assume 1.
3. FINANCIALS: Carefully identify the Subtotal, Tax, and true Grand Total. Do not confuse account balances or membership numbers with the Grand Total or prices.
4. MATH VERIFICATION: (Sum of line item totals) + Tax + Tip must closely match the Grand Total.
5. Transcription must be literal. Use null if a field is missing. Do not guess.

### RULE: HORIZONTAL ANCHORING
For every line item, you must read strictly from LEFT to RIGHT.
- Identify the Item Name on the left.
- Trace a straight horizontal line to the far right to find the Price.
- If the price does not align horizontally with the item name, DO NOT extract it.
- Explicitly look for the 'Paper Bag Fee' ($0.50) located just above the Tax.

### AUDIT RULES:
1. DECOMPOSE MULTIPLIERS: If you see '6 @ 19.99' or '2 x 5.00', you MUST extract Qty: 6 and Unit Price: 19.99. Do not just put '1' as the quantity.
2. THE FOOTER AUDIT: After the line items, check for 'Tax', 'Paper Bag Fee', or 'Surcharges'. These MUST be included in the financials.
3. ABSOLUTE TOTAL: Find the Grand Total at the bottom. It is the final amount paid.

### OUTPUT SCHEMA (STRICT JSON):
{
  "merchant": { "name": "", "address": "", "phone": "" },
  "metadata": { "date": "", "time": "", "currency": "" },
  "line_items": [
    { "item": "", "qty": 1, "price": 0.00, "total": 0.00, "box_2d": [0, 0, 0, 0] }
  ],
  "financials": {
    "subtotal": 0.00,
    "tax": 0.00,
    "tip": 0.00,
    "grand_total": 0.00
  },
  "flags": {
    "arithmetic_match": true,
    "contains_alcohol": false,
    "unreadable_sections": []
  }
}
"""

In [6]:
from decimal import Decimal

def validate_receipt_sum(line_items, tax, tip, printed_total):
    """
    Deterministic tool to verify if line items match the printed total.
    """

    # Use Decimal for financial precision (avoids float errors)
    item_total = sum(Decimal(str(item['total'])) for item in line_items)
    calculated_grand_total = item_total + Decimal(str(tax)) + Decimal(str(tip))

    print(f"DEBUG: The VLM thinks the Printed Total is: {printed_total}")
    print(f"DEBUG: My Python sum calculated: {calculated_grand_total}")

    difference = calculated_grand_total - Decimal(str(printed_total))

    return {
        "is_match": abs(difference) < Decimal('0.01'),
        "calculated_total": float(calculated_grand_total),
        "difference": float(difference),
        "status": "Verified" if abs(difference) < Decimal('0.01') else "Discrepancy Detected"
    }

In [10]:
from PIL import Image


def pad_to_square(image_path, output_path=None):
    img = Image.open(image_path)
    width, height = img.size
    max_dim = max(width, height)
    new_img = Image.new('RGB', (max_dim, max_dim), (255, 255, 255))
    left = (max_dim - width) // 2
    top = (max_dim - height) // 2
    new_img.paste(img, (left, top))
    if output_path:
        new_img.save(output_path)
    return new_img

In [11]:
import json

def ask_vlm(image_url, prompt, processor, model):
    messages = [
        {
            "role": "user", "content": [
                {"type": "image", "url": image_url},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]
    outputs = model.generate(**inputs, max_new_tokens=2048)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    parsed_content = processor.parse_response(response)['content']

    json_match = re.search(r'\{.*\}', parsed_content, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            return None
    return None

In [9]:
import re
import os

# --- Main Agent Logic ---
class ReceiptExtractionAgent:
    def __init__(self, processor, model, max_retries=1):
        self.processor = processor
        self.model = model
        self.max_retries = max_retries
        self.evaluation_log = []

    def log_step(self, step_type, content):
        print(f"[{step_type.upper()}] {content}")
        self.evaluation_log.append({"step": step_type, "content": content})

    def run(self, original_image_path, base_prompt):
        attempt = 0
        padded_image_path = "/tmp/padded_receipt.jpg"
        pad_to_square(original_image_path, padded_image_path)

        current_prompt = base_prompt
        receipt_data = None

        while attempt <= self.max_retries:
            self.log_step("Thought", f"Attempt {attempt + 1}: Starting extraction process for full receipt.")

            # PLAN
            self.log_step("Action", "Calling ask_vlm to extract receipt data from padded image (PLAN)")
            receipt_data = ask_vlm(padded_image_path, current_prompt, self.processor, self.model)

            if not receipt_data:
                self.log_step("Observation", "Failed to extract valid JSON from VLM.")
                attempt += 1
                continue

            # EXECUTE
            self.log_step("Action", "Passing extracted data to validate_receipt_sum (EXECUTE)")
            financials = receipt_data.get('financials', {})
            current_line_items = receipt_data.get('line_items', [])

            # Print extracted items table before verification
            print("\n--- Extracted Line Items ---")
            print(f"{'Item':<45} | {'Qty':<5} | {'Price':<8} | {'Total':<8}")
            print("-" * 75)
            for item in current_line_items:
                name = str(item.get('item', ''))[:45]
                qty = str(item.get('qty', ''))
                price = str(item.get('price', ''))
                total = str(item.get('total', ''))
                print(f"{name:<45} | {qty:<5} | {price:<8} | {total:<8}")
            print("-" * 75 + "\n")

            tax = financials.get('tax', 0.0)
            tip = financials.get('tip', 0.0)
            grand_total = financials.get('grand_total', 0.0)

            validation_result = validate_receipt_sum(current_line_items, tax, tip, grand_total)
            self.log_step("Observation", f"Validation Result: {json.dumps(validation_result)}")

            # CRITIC
            self.log_step("Thought", "Evaluating execution results (CRITIC)")
            if validation_result["is_match"]:
                self.log_step("Observation", "Math matches perfectly. Task complete.")
                break
            else:
                diff = validation_result["difference"]
                self.log_step("Thought", f"Discrepancy detected ({diff}).")

                if attempt < self.max_retries:
                    self.log_step("Action", f"Retrying extraction... (Retries left: {self.max_retries - attempt})")
                    current_prompt = base_prompt + f"\n\nCRITIC FEEDBACK: Your previous extraction was off by {diff}. Please look again and find the missing items or correct the discrepancy."
                    attempt += 1
                else:
                    self.log_step("Observation", "Max retries reached. Flagging for 'Manual Human Review'.")
                    if 'flags' not in receipt_data:
                        receipt_data['flags'] = {}
                    receipt_data['flags']['manual_review'] = True
                    break

        return receipt_data

# --- Execution ---
image_path = "/content/data/Receipt - Carnegie Mellon University Store - July 10, 2025.jpg"
agent = ReceiptExtractionAgent(processor, model, max_retries=1)
final_data = agent.run(image_path, PROMPT)

print("\n--- Final Agent Execution Log ---")
print(json.dumps(agent.evaluation_log, indent=2))


[THOUGHT] Attempt 1: Starting extraction process for full receipt.
[ACTION] Calling ask_vlm to extract receipt data from padded image (PLAN)
[ACTION] Passing extracted data to validate_receipt_sum (EXECUTE)

--- Extracted Line Items ---
Item                                          | Qty   | Price    | Total   
---------------------------------------------------------------------------
PLUSH SCOTTY DOG                              | 1     | 119.94   | 119.94  
JACKET JOURNEY LDS                            | 1     | 69.49    | 69.49   
HOOK SCOTTY SHIELD CM                         | 1     | 34.49    | 34.49   
HODD SCOTTY SHIELD 1991                       | 1     | 64.99    | 64.99   
---------------------------------------------------------------------------

DEBUG: The VLM thinks the Printed Total is: 307.81
DEBUG: My Python sum calculated: 297.31
[OBSERVATION] Validation Result: {"is_match": false, "calculated_total": 297.31, "difference": -10.5, "status": "Discrepancy Detected"}
[TH